# Omarchy Nano: Qwen3.5-2B QLoRA fine-tuning

This is the cleaned reproducibility notebook for the `omarchy-nano` run. It is designed for a Google Colab GPU and keeps source checkouts, datasets, adapters, and GGUF exports in Google Drive.

No Hugging Face token, model weight, checkpoint, or generated model artifact is embedded in this notebook or the GitHub repository. Store `HF_TOKEN` in Colab Secrets and use the optional upload cell at the end.

The completed reference run used 4-bit QLoRA on a Tesla T4 for two epochs. Its recorded final losses are preserved in `evaluation/results.json`; rerunning this notebook may produce slightly different values.

## 1. Install the pinned training stack

Run this cell in a fresh Colab runtime. Restart the runtime if Colab asks after installation.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/EF-Code/omarchy-nano.git"
REPO_DIR = Path("/content/omarchy-nano")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements-colab.txt")], check=True)
print(f"Repository: {REPO_DIR}")

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/omarchy-nano")
SOURCE_DIR = PROJECT_DIR / "source" / "omarchy"
DATA_DIR = PROJECT_DIR / "data"
EXPORT_DIR = PROJECT_DIR / "exports"
ADAPTER_DIR = EXPORT_DIR / "omarchy-nano-qwen3.5-2b-qlora"
GGUF_DIR = EXPORT_DIR / "omarchy-nano-qwen3.5-2b-gguf-q4_k_m_gguf"
for directory in (SOURCE_DIR.parent, DATA_DIR, EXPORT_DIR, ADAPTER_DIR):
    directory.mkdir(parents=True, exist_ok=True)

BASE_MODEL = "Qwen/Qwen3.5-2B-Base"
OMARCHY_REPO = "https://github.com/omacom/omarchy.git"
OMARCHY_REF = "quattro"
OMARCHY_COMMIT = "d3d23fdddef846ebb98b52122a6ece66211c0daf"
SEED = 3407
MAX_SEQUENCE_LENGTH = 512
HF_REPO = "NewSonnet/omarchy-nano"
print({"base_model": BASE_MODEL, "project_dir": str(PROJECT_DIR), "seed": SEED})

## 2. Fetch the exact Omarchy source snapshot

The commit is recorded in `data/PROVENANCE.json`. A moving branch should not be used for a reproducible rebuild.

In [ ]:
if not (SOURCE_DIR / ".git").exists():
    subprocess.run(["git", "clone", "--filter=blob:none", OMARCHY_REPO, str(SOURCE_DIR)], check=True)
subprocess.run(["git", "-C", str(SOURCE_DIR), "fetch", "--depth", "1", "origin", OMARCHY_COMMIT], check=True)
subprocess.run(["git", "-C", str(SOURCE_DIR), "checkout", "--detach", OMARCHY_COMMIT], check=True)
actual_commit = subprocess.check_output(["git", "-C", str(SOURCE_DIR), "rev-parse", "HEAD"], text=True).strip()
assert actual_commit == OMARCHY_COMMIT, (actual_commit, OMARCHY_COMMIT)
print(f"Checked out {actual_commit} from {OMARCHY_REF}")

## 3. Build or reuse the sanitized JSONL dataset

If `data/train.jsonl` and `data/eval.jsonl` already exist in Drive, they are reused. Otherwise the checked-in preprocessing script creates a text-only baseline from the source snapshot. Images are replaced by `[image omitted]`, and the split is document-level to reduce leakage.

In [ ]:
sys.path.insert(0, str(REPO_DIR))
from scripts.prepare_omarchy_data import iter_markdown_files, make_examples, split_documents, write_jsonl

TRAIN_PATH = DATA_DIR / "train.jsonl"
EVAL_PATH = DATA_DIR / "eval.jsonl"
if not (TRAIN_PATH.exists() and EVAL_PATH.exists()):
    markdown_files = iter_markdown_files(SOURCE_DIR)
    train_files, eval_files = split_documents(markdown_files, eval_fraction=0.125, seed=SEED)
    write_jsonl(TRAIN_PATH, make_examples(train_files, SOURCE_DIR, max_chars=3200))
    write_jsonl(EVAL_PATH, make_examples(eval_files, SOURCE_DIR, max_chars=3200))
    print(f"Generated {TRAIN_PATH} and {EVAL_PATH}")
else:
    print("Reusing the existing Drive dataset")

print("train bytes:", TRAIN_PATH.stat().st_size)
print("eval bytes:", EVAL_PATH.stat().st_size)

In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files={"train": str(TRAIN_PATH), "eval": str(EVAL_PATH)})
print(dataset)
print(dataset["train"][0]["messages"][0]["content"][:500])

## 4. Load the base model and attach QLoRA

The T4 path uses 4-bit loading and fp16. LoRA adapters are saved to Drive, not to the GitHub checkout.

In [ ]:
import torch
from transformers import set_seed
from unsloth import FastLanguageModel

set_seed(SEED)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQUENCE_LENGTH,
    load_in_4bit=True,
    dtype=None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing=False,
    random_state=SEED,
    use_rslora=False,
)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
model.print_trainable_parameters()

In [ ]:
def formatting_func(examples):
    if "messages" in examples:
        return [
            tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
            for messages in examples["messages"]
        ]
    return examples["text"]

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(formatting_func(dataset["train"][0:1])[0][:700])

## 5. Train

The settings below reproduce the reference run: two epochs, rank-16 LoRA, max length 512, effective batch size 8, learning rate `2e-4`, and 8-bit AdamW.

In [ ]:
from trl import SFTConfig, SFTTrainer

supports_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
training_args = SFTConfig(
    output_dir=str(PROJECT_DIR / "checkpoints"),
    num_train_epochs=2,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_ratio=0.1,
    logging_steps=1,
    eval_strategy="steps",
    eval_steps=25,
    save_strategy="no",
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    fp16=not supports_bf16,
    bf16=supports_bf16,
    max_length=MAX_SEQUENCE_LENGTH,
    packing=False,
    gradient_checkpointing=False,
    seed=SEED,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["eval"],
    processing_class=tokenizer,
    formatting_func=formatting_func,
)
train_result = trainer.train()
eval_metrics = trainer.evaluate()
print(train_result.metrics)
print(eval_metrics)

In [ ]:
metrics = {
    "base_model": BASE_MODEL,
    "source_commit": OMARCHY_COMMIT,
    "train_metrics": train_result.metrics,
    "eval_metrics": eval_metrics,
}
metrics_path = EXPORT_DIR / "run_metrics.json"
metrics_path.write_text(json.dumps(metrics, indent=2, default=float) + "\n", encoding="utf-8")
print(metrics_path)

## 6. Save the adapter and optionally export GGUF

Both operations below write only under Google Drive. Do not point either output path at the GitHub checkout.

In [ ]:
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
print(f"Adapter saved to {ADAPTER_DIR}")

In [ ]:
EXPORT_GGUF = False  # Set True only when you want to regenerate the Drive export.
if EXPORT_GGUF:
    GGUF_DIR.mkdir(parents=True, exist_ok=True)
    model.save_pretrained_gguf(str(GGUF_DIR), tokenizer, quantization_method="q4_k_m")
    print(f"GGUF export saved to {GGUF_DIR}")
else:
    print("GGUF export skipped")

## 7. Optional direct upload to Hugging Face

Create a Colab Secret named `HF_TOKEN` with a write-scoped token before running this cell. The token is read from the secret store and is never printed or written to disk.

In [ ]:
from google.colab import userdata
from huggingface_hub import HfApi

hf_token = userdata.get("HF_TOKEN")
if not hf_token:
    raise RuntimeError("Add a write-scoped HF_TOKEN Colab Secret before uploading.")
api = HfApi(token=hf_token)
api.create_repo(repo_id=HF_REPO, repo_type="model", exist_ok=True)
api.upload_folder(
    repo_id=HF_REPO,
    repo_type="model",
    folder_path=str(ADAPTER_DIR),
    path_in_repo="adapter",
    commit_message="Upload Omarchy Nano QLoRA adapter",
)
if GGUF_DIR.exists():
    api.upload_folder(
        repo_id=HF_REPO,
        repo_type="model",
        folder_path=str(GGUF_DIR),
        path_in_repo="gguf-q4_k_m",
        commit_message="Upload Omarchy Nano GGUF export",
    )
print(f"Uploaded artifacts to https://huggingface.co/{HF_REPO}")